# Hospital Readmission Prediction Project: Statistical Analysis


### **Author:**  Ari M.


### **Purpose & Goal:**
### Validate some of the patterns from the EDA before moving into modeling.
### Compare readmitted vs non-readmitted patients
### Test whether some of the differences are statistically significant
### Figure out which variables might actually matter later in prediction


### **Datasets used:** diabetic_features.csv, diabetic_cleaned.csv

## Plan

For this notebook I mainly want to sanity check some of the EDA findings statistically.

I'll use:
- chi-square tests for categorical variables
- t-tests for numerical variables

Mostly trying to narrow down which features actually seem connected to 30-day readmission before getting deeper into modeling.

In [1]:
import pandas as pd
import numpy as np

In [2]:
import kagglehub

# Download latest version
feature_path = kagglehub.dataset_download("datascienceam/diabetic-features-csv")
raw_path = kagglehub.dataset_download("datascienceam/diabetic-cleaned-csv")

print("Path to dataset files:", feature_path)
print("Path to dataset files:", raw_path)

Path to dataset files: /kaggle/input/datasets/datascienceam/diabetic-features-csv
Path to dataset files: /kaggle/input/datasets/datascienceam/diabetic-cleaned-csv


In [3]:
# On local system
# Encoded version for modeling later
#df = pd.read_csv("../data/processed/diabetic_features.csv")

# Original cleaned dataset for easier categorical testing
#df_raw = pd.read_csv("../data/processed/diabetic_cleaned.csv",low_memory=False)

# On Kaggle
import os

feature_file_path = os.path.join(feature_path, "diabetic_features.csv")
df = pd.read_csv(feature_file_path, low_memory=False)

raw_file_path = os.path.join(raw_path, "diabetic_cleaned.csv")
df_raw = pd.read_csv(raw_file_path, low_memory=False)

df.head()

,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,...,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady,change_No,diabetesMed_Yes,readmitted_30
0,6,25,1,1,41,0,1,0,0,0,...,True,False,False,False,False,False,False,True,False,0
1,1,1,7,3,59,0,18,0,0,0,...,True,False,False,False,False,False,False,False,True,0
2,1,1,7,2,11,5,13,2,0,1,...,True,False,False,False,False,False,False,True,True,0
3,1,1,7,2,44,1,16,0,0,0,...,True,False,False,False,False,False,False,False,True,0
4,1,1,7,1,51,0,8,0,0,0,...,True,False,False,False,False,False,False,False,True,0


In [4]:
target = "readmitted_30"

y = df[target]
X = df.drop(columns=[target])

print(y.value_counts(normalize=True))

readmitted_30
0    0.888401
1    0.111599
Name: proportion, dtype: float64


## Quick Note on the Target Distribution

The dataset is pretty imbalanced:
- ~11% readmitted
- ~89% not readmitted

So later on, accuracy alone probably won't be very useful.

Also worth remembering that with a dataset this large, even small differences can end up looking statistically significant.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (81412, 2417)
Test shape: (20354, 2417)


## Train/Test Split

I split the data pretty early mainly to avoid leakage later on.

For the statistical testing below, I'm using the training data where possible so I don't accidentally use information from the test set while making modeling decisions.

In [6]:
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_medications",
    "number_inpatient",
    "number_emergency"
]

df_train = X_train.copy()
df_train["readmitted_30"] = y_train

df_train.groupby("readmitted_30")[num_cols].mean()

,time_in_hospital,num_lab_procedures,num_medications,number_inpatient,number_emergency
readmitted_30,,,,,
0,4.347067,42.954567,15.911069,0.561195,0.180253
1,4.759410,44.190073,16.896214,1.225622,0.358243


## Initial Observations

The biggest difference here is definitely prior inpatient visits.

Readmitted patients also seem to:
- stay slightly longer
- take more medications on average
- have more emergency visits

The gaps for some of these variables are smaller than I expected though.

At first glance, inpatient history looks like one of the stronger signals so far.

### Small Reflection

One thing that surprised me was how much larger the inpatient difference was compared to emergency visits.

I originally expected emergency utilization to stand out more strongly.

Not really sure yet whether inpatient history is capturing actual disease severity or just repeated interaction with the healthcare system in general. Might become clearer once I start modeling.

In [7]:
from scipy.stats import chi2_contingency

def chi_square_test_raw(feature):
    contingency_table = pd.crosstab(df_raw[feature], df_raw["readmitted_30"])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    print(f"{feature} → p-value: {p}")

In [8]:
chi_square_test_raw("gender")
chi_square_test_raw("race")
chi_square_test_raw("diabetesMed")

gender → p-value: 0.5386832905074449
race → p-value: 0.17748772174247665
diabetesMed → p-value: 5.5651160886737224e-18


## Chi-Square Test Results

`diabetesMed` came back strongly significant, while gender and race didn't show much relationship here.

Not super surprising honestly. Medication-related variables probably connect more directly to disease management than demographics alone.

Still, these tests only show association, not causation.

### Why I Used the Raw Dataset Here

For the chi-square tests I used the non-encoded dataset so I could test the full categorical variables directly instead of individual one-hot encoded columns.

Makes the results a lot easier to interpret.

### Limitation
One limitation here is that these tests are all independent, so some significant results could still happen by chance.

Also important: statistical significance definitely doesn't mean causation.

In [9]:
from scipy.stats import ttest_ind

def t_test(feature):
    group0 = df_train[df_train["readmitted_30"] == 0][feature]
    group1 = df_train[df_train["readmitted_30"] == 1][feature]
    
    stat, p = ttest_ind(group0, group1, equal_var=False)
    print(f"{feature} → p-value: {p}")
for col in num_cols:
    t_test(col)

time_in_hospital → p-value: 2.4288220751820047e-34
num_lab_procedures → p-value: 9.278106169511004e-09
num_medications → p-value: 2.0784364311332028e-27
number_inpatient → p-value: 7.830965637332039e-209
number_emergency → p-value: 2.8783129982744454e-31


## T-Test Results

Pretty much every numerical variable tested came back statistically significant.

But with ~100k rows, that's not necessarily saying much by itself.

The inpatient variable still stands out though, both statistically and in terms of the actual gap between groups.

Some of the other variables are probably significant mostly because the dataset is so large.

Would be useful to check effect sizes later too instead of relying only on p-values.

### Note on Multiple Testing

Since I'm testing several variables independently, there's always a chance that a few significant results are happening randomly.

I didn't apply any correction yet since this is still exploratory analysis, but that would matter more in a stricter statistical setting.

## Things I'll Need to Handle During Modeling

A few things already look like they'll matter later:

- class imbalance (~11% positive class)
- lots of categorical variables
- high-cardinality diagnosis columns
- missing values that might actually contain useful signal

The diagnosis features especially feel a bit messy right now and probably need more feature engineering.

## Final Notes

A few variables seem consistently tied to readmission, especially prior inpatient visits.

Length of stay, medications, and emergency visits also show differences between groups, although some of those effects look fairly small in practice.

A couple demographic variables didn't show much relationship here, at least from these initial tests.

The dataset itself also has a few annoying challenges:
- lots of categorical features
- some messy diagnosis columns
- missingness that may or may not be informative

Next step will probably be building a simple baseline model first (likely logistic regression) before trying anything more complex.

I also want to spend more time on diagnosis-related feature engineering since that feels like one of the weaker parts of the current dataset setup.